# Session 2: measure similarity, predict, then test honestly
**A. Real-world problem.** Session 1 compared 5 people by hand. A real service has
thousands of users, each with a few dozen ratings. How do we score "similar taste" with a
formula, turn it into a predicted rating, and know that the predictions are any good?

**Plan.** (1) make a seeded synthetic ratings matrix from hidden taste factors, (2) cosine
and Pearson similarity on co-rated items, (3) predict as a similarity-weighted average,
user-based and item-based, (4) hide 20% of known ratings and score with RMSE and hit rate.
All data is synthetic; the "tastes" are random numbers.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a synthetic world with hidden tastes
Every user has 3 hidden taste numbers and every item has 3 hidden "flavour" numbers. The
true liking is `3.2 + user bias + item bias + 0.8 * (taste . flavour)`, plus a little noise,
rounded to whole stars 1..5. Only some cells are *observed*, and popular (high item bias)
items are observed more often, as in real life.

In [ ]:
def make_ratings(n_users=100, n_items=60, factors=3, density=0.35, seed=0):
    """Return (R, M): R is the full 1..5 matrix, M marks which cells were observed."""
    rng = np.random.default_rng(seed)
    U = rng.normal(size=(n_users, factors))
    V = rng.normal(size=(n_items, factors))
    user_bias = rng.normal(0, 0.3, n_users)
    item_bias = rng.normal(0, 0.5, n_items)
    raw = 3.2 + user_bias[:, None] + item_bias[None, :] + 0.8 * U @ V.T + rng.normal(0, 0.3, (n_users, n_items))
    R = np.clip(np.rint(raw), 1, 5)
    w = np.exp(1.2 * item_bias)
    p = np.clip(density * n_items * w / w.sum(), 0, 0.9)      # exposure of each item
    M = rng.random((n_users, n_items)) < p[None, :]
    return R, M


def split_observed(M, test_fraction=0.2, seed=1):
    """Hide a random share of the observed cells. Returns (train_mask, test_mask)."""
    rng = np.random.default_rng(seed)
    test = M & (rng.random(M.shape) < test_fraction)
    return M & ~test, test

## Step 2: similarity on co-rated items
For users a and b take only the items both rated, giving rating lists x and y.

    cosine  = (x . y) / (|x| |y|)
    pearson = the same formula after subtracting each list's own mean

Pearson removes "this user rates everything high". With plain 1..5 ratings, cosine is
always positive and barely tells people apart. We need at least `min_common` shared items.

In [ ]:
def similarity(R, M, method="pearson", min_common=3):
    """Square matrix of similarities between the rows of R (zero where too little is shared)."""
    n = R.shape[0]
    S = np.zeros((n, n))
    for a in range(n):
        for b in range(a + 1, n):
            both = M[a] & M[b]
            if both.sum() < min_common:
                continue
            x, y = R[a, both], R[b, both]
            if method == "pearson":
                x, y = x - x.mean(), y - y.mean()
            den = np.linalg.norm(x) * np.linalg.norm(y)
            S[a, b] = S[b, a] = (x @ y) / den if den > 0 else 0.0
    return S

## Step 3: predict = own average + similarity-weighted opinions of neighbours

    pred(u, i) = mean_u + sum_v s(u,v) * (r(v,i) - mean_v) / sum_v |s(u,v)|

The sums run over the `k` most similar users v (positive similarity) who rated item i.
If nobody qualifies we return the user's own mean. For **item-based** prediction we run
the very same code on the transposed table (items play the role of users).

In [ ]:
def top_k_neighbours(S, k):
    """Keep only the k largest positive similarities in each row."""
    out = np.zeros_like(S)
    for a in range(S.shape[0]):
        row = S[a].copy()
        row[a] = 0.0
        idx = np.argsort(row)[::-1][:k]
        idx = idx[row[idx] > 0]
        out[a, idx] = row[idx]
    return out


def predict_all(R, M, S, k=20):
    """Predicted rating for every cell of R, from the observed cells in M."""
    counts = np.maximum(M.sum(axis=1), 1)
    means = (R * M).sum(axis=1) / counts
    centred = (R - means[:, None]) * M
    Sk = top_k_neighbours(S, k)
    num = Sk @ centred
    den = np.abs(Sk) @ M.astype(float)
    adjust = np.where(den > 0, num / np.where(den > 0, den, 1.0), 0.0)
    return np.clip(means[:, None] + adjust, 1, 5)


def predict_user_based(R, M, k=20, method="pearson"):
    return predict_all(R, M, similarity(R, M, method), k)


def predict_item_based(R, M, k=20, method="pearson"):
    return predict_all(R.T, M.T, similarity(R.T, M.T, method), k).T

## Step 4: evaluate by hiding known ratings
* **RMSE** = sqrt(mean((prediction - truth)^2)) over the hidden cells, in stars. Smaller is better.
* **Hit rate at N**: for each user who has at least one hidden item rated 4 or 5 ("liked"),
  rank all items they have not rated in the training data; a hit means a liked hidden item
  appears in the top N (we use N = 5).

In [ ]:
def rmse(pred, R, mask):
    return float(np.sqrt(np.mean((pred[mask] - R[mask]) ** 2)))


def global_mean_prediction(R, train):
    return np.full(R.shape, R[train].mean())


def item_mean_prediction(R, train):
    counts = np.maximum(train.sum(axis=0), 1)
    means = np.where(train.sum(axis=0) > 0, (R * train).sum(axis=0) / counts, R[train].mean())
    return np.tile(means, (R.shape[0], 1))


def hit_rate(score, R, train, test, n=5, liked=4):
    """Share of eligible users with a liked hidden item in their top-n list."""
    hits, users = 0, 0
    for u in range(R.shape[0]):
        target = test[u] & (R[u] >= liked)
        if not target.any():
            continue
        users += 1
        s = np.where(train[u], -np.inf, score[u])
        top = np.argsort(-s)[:n]
        hits += bool(target[top].any())
    return hits / users


def random_hit_rate(R, train, test, n=5, liked=4, trials=20, seed=0):
    rng = np.random.default_rng(seed)
    return float(np.mean([hit_rate(rng.random(R.shape), R, train, test, n, liked) for _ in range(trials)]))


def popularity_score(train):
    """How many training ratings each item has, repeated for every user."""
    return np.tile(train.sum(axis=0).astype(float), (train.shape[0], 1))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    R, M = make_ratings()
    train, test = split_observed(M)
    print(f"{R.shape[0]} users x {R.shape[1]} items, {M.sum()} observed ratings, {test.sum()} hidden for testing")
    results = {}
    results["global mean"] = rmse(global_mean_prediction(R, train), R, test)
    results["item mean"] = rmse(item_mean_prediction(R, train), R, test)
    Pu = predict_user_based(R, train)
    Pi = predict_item_based(R, train)
    results["user-based (Pearson)"] = rmse(Pu, R, test)
    results["item-based (Pearson)"] = rmse(Pi, R, test)
    results["user-based (cosine)"] = rmse(predict_user_based(R, train, method="cosine"), R, test)
    for name, v in results.items():
        print(f"RMSE {name:22s} {v:.3f} stars")
    print(f"hit rate@5  random {random_hit_rate(R, train, test):.3f}  popularity {hit_rate(popularity_score(train), R, train, test):.3f}"
          f"  user-based {hit_rate(Pu, R, train, test):.3f}  item-based {hit_rate(Pi, R, train, test):.3f}")

    ks = [1, 2, 5, 10, 20, 40, 80]
    S = similarity(R, train)
    curve = [rmse(predict_all(R, train, S, k), R, test) for k in ks]
    print("RMSE versus k:", [f"{k}:{c:.3f}" for k, c in zip(ks, curve)])

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].hist(similarity(R, train, "cosine")[np.triu_indices(100, 1)], bins=30, alpha=0.6, label="cosine")
    axes[0].hist(S[np.triu_indices(100, 1)], bins=30, alpha=0.6, label="Pearson")
    axes[0].set_xlabel("similarity between two users (no unit)")
    axes[0].set_ylabel("number of user pairs")
    axes[0].set_title("Cosine is bunched near 1; Pearson spreads out")
    axes[0].legend()
    names = list(results)
    axes[1].barh(names, [results[n] for n in names], color="#2a9d8f")
    axes[1].set_xlabel("test RMSE (stars)")
    axes[1].set_title("Hidden-rating error by method")
    axes[2].plot(ks, curve, "o-")
    axes[2].set_xscale("log")
    axes[2].set_xlabel("neighbours k")
    axes[2].set_ylabel("test RMSE (stars)")
    axes[2].set_title("User-based error versus k")
    for a in axes:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "similarity_and_error.png", dpi=120)

## Where this stops
Neighbour methods look at raw pairs of users. Session 3 learns a small set of hidden tastes
directly (matrix factorisation) and also faces cold start and feedback loops.

## Coding tasks
1. **Easy.** Print `similarity(R, M)` for users 0 and 1 of the synthetic world with `method="cosine"` and `"pearson"` and explain why the two numbers differ.
2. **Stretch.** Change `min_common` from 3 to 5 in `similarity` and see how the user-based RMSE changes; explain the direction.
3. **Explain.** Explain why item-based prediction needs no new code in `lesson.py`, and what the rows and columns of the transposed table mean.

## Assessment
- **A. Conceptual.** Why does Pearson similarity separate users better than cosine when all ratings are between 1 and 5? What does a negative Pearson mean?
- **B. Numerical.** Two users share 3 items. User x rated them 5, 3, 1 and user y rated them 4, 4, 1. Compute the cosine and the Pearson similarity (no unit).
- **C. Graph and simulation interpretation.** In the right panel, the error falls from 1.229 at k = 1. Roughly where is the best k, and why does the error rise again for very large k?
- **D. Design and optimisation challenge.** Design: the checkpoint reports hit rate at 5 of 0.271 for random, 0.747 for popularity, 0.621 for user-based and 0.678 for item-based, but the RMSE of user-based is much better than popularity-like baselines. Which method would you ship for a 'top 5 picks' row, and what would you change in the test before trusting the answer?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    nan = np.nan

    # --- worked example by hand: 4 users x 5 items, user 0 has not rated item 4
    T = np.array([[5, 3, 4, 4, nan],
                  [3, 1, 2, 2, 3],
                  [1, 5, 2, 4, 2],
                  [4, 3, 4, 3, 5]], dtype=float)
    M = ~np.isnan(T)
    R = np.nan_to_num(T)
    cos = L.similarity(R, M, "cosine")
    pea = L.similarity(R, M, "pearson")
    hand_cos = 34 / (np.sqrt(25 + 9 + 16 + 16) * np.sqrt(9 + 1 + 4 + 4))
    checks.append((f"cosine of users 0 and 1 on 4 shared items is 34/(8.124*4.243) = 0.986 (got {cos[0, 1]:.3f})", abs(cos[0, 1] - hand_cos) < 1e-9 and abs(cos[0, 1] - 0.986) < 5e-4))
    checks.append(("Pearson of users 0 and 1 is exactly 1 (same pattern, shifted)", abs(pea[0, 1] - 1.0) < 1e-12))
    checks.append(("Pearson of users 0 and 2 is -4/(1.414*3.162) = -0.894", abs(pea[0, 2] + 0.894) < 5e-4))
    checks.append(("Pearson of users 0 and 3 is 0.707", abs(pea[0, 3] - 0.7071) < 5e-4))
    checks.append(("similarity matrix is symmetric", np.allclose(pea, pea.T)))
    pred = L.predict_all(R, M, pea, k=3)
    # mean_0 = 4; user 1 contributes 1.0 * (3 - 2.2); user 3 contributes 0.7071 * (5 - 3.8); user 2 is negative and ignored
    hand = 4 + (1.0 * 0.8 + 0.7071 * 1.2) / (1.0 + 0.7071)
    checks.append((f"worked example: prediction for user 0, item 4 is 4 + (0.8 + 0.8485)/1.7071 = 4.966 (got {pred[0, 4]:.3f})", abs(pred[0, 4] - hand) < 1e-3 and abs(pred[0, 4] - 4.966) < 5e-4))
    pred1 = L.predict_all(R, M, pea, k=1)
    checks.append(("with k = 1 only the closest user counts: 4 + 0.8 = 4.8", abs(pred1[0, 4] - 4.8) < 1e-9))
    checks.append(("predictions stay inside 1..5", bool(pred.min() >= 1 and pred.max() <= 5)))

    # --- edges
    few = np.array([[5, 4, nan, nan], [4, 5, 3, nan]], dtype=float)
    Mf = ~np.isnan(few)
    checks.append(("edge: only 2 shared items gives similarity 0 (below min_common = 3)", L.similarity(np.nan_to_num(few), Mf)[0, 1] == 0.0))
    lone = L.predict_all(np.nan_to_num(few), Mf, L.similarity(np.nan_to_num(few), Mf), k=5)
    checks.append(("edge: no neighbours means the user's own mean (4.5)", abs(lone[0, 2] - 4.5) < 1e-12))
    flat = np.array([[3, 3, 3, 3], [1, 2, 3, 4]], dtype=float)
    checks.append(("edge: a user who rates everything 3 gives Pearson 0, not NaN", L.similarity(flat, np.ones_like(flat, bool))[0, 1] == 0.0))
    ii = L.predict_item_based(R, M, k=3)
    checks.append(("item-based = user-based code on the transposed table", np.allclose(ii, L.predict_all(R.T, M.T, L.similarity(R.T, M.T), 3).T)))

    # --- assessment answers and the app challenge
    X = np.array([[5, 3, 1], [4, 4, 1]], dtype=float)
    MX = np.ones_like(X, dtype=bool)
    checks.append(("exercise B: cosine 0.971 and Pearson 0.866 for (5,3,1) and (4,4,1)",
                   round(L.similarity(X, MX, "cosine")[0, 1], 3) == 0.971 and round(L.similarity(X, MX)[0, 1], 3) == 0.866))
    Y = np.array([[5, 3, 4, 4], [3, 5, 4, 4]], dtype=float)
    MY = np.ones_like(Y, dtype=bool)
    checks.append(("app challenge: B = (3,5,4,4) has cosine 0.939 but Pearson -1",
                   round(L.similarity(Y, MY, "cosine")[0, 1], 3) == 0.939 and abs(L.similarity(Y, MY)[0, 1] + 1) < 1e-9))

    # --- the synthetic world and honest evaluation
    R, M = L.make_ratings()
    train, test = L.split_observed(M)
    checks.append(("seeded world is reproducible", np.array_equal(R, L.make_ratings()[0])))
    checks.append(("world: 100 users x 60 items with 2034 observed ratings", R.shape == (100, 60) and int(M.sum()) == 2034))
    checks.append(("2034 observed ratings split into 434 hidden and 1600 for training", int(test.sum()) == 434 and int(train.sum()) == 1600))
    checks.append(("train and test cells never overlap", not (train & test).any()))
    checks.append(("ratings are whole stars 1..5", bool(set(np.unique(R)) <= {1.0, 2.0, 3.0, 4.0, 5.0})))
    gm = L.rmse(L.global_mean_prediction(R, train), R, test)
    im = L.rmse(L.item_mean_prediction(R, train), R, test)
    Pu = L.predict_user_based(R, train)
    Pi = L.predict_item_based(R, train)
    Pc = L.predict_user_based(R, train, method="cosine")
    ru, ri, rc = (L.rmse(p, R, test) for p in (Pu, Pi, Pc))
    checks.append((f"RMSE: global mean 1.242 > item mean 1.179 (got {gm:.3f}, {im:.3f})", round(gm, 3) == 1.242 and round(im, 3) == 1.179))
    checks.append((f"RMSE: user-based Pearson 0.873, item-based 0.906, cosine 0.937 (got {ru:.3f}, {ri:.3f}, {rc:.3f})", (round(ru, 3), round(ri, 3), round(rc, 3)) == (0.873, 0.906, 0.937)))
    checks.append(("neighbour methods beat the item-mean baseline; Pearson beats cosine", ru < im and ri < im and ru < rc))
    # no leakage: changing the hidden ratings must not change the predictions
    R2 = R.copy()
    R2[test] = 1.0
    checks.append(("no leakage: hidden ratings do not influence predictions", np.allclose(L.predict_user_based(R2, train), Pu)))
    # the cosine values are all positive and bunched; Pearson spreads out
    iu = np.triu_indices(100, 1)
    Sc, Sp = L.similarity(R, train, "cosine")[iu], L.similarity(R, train)[iu]
    checks.append(("cosine on 1..5 ratings is never negative; Pearson takes both signs", bool(Sc.min() >= 0) and bool(Sp.min() < 0 < Sp.max())))
    checks.append(("Pearson spread is larger than cosine spread", Sp.std() > Sc.std()))
    ks = [1, 2, 5, 10, 20, 40, 80]
    S = L.similarity(R, train)
    curve = [L.rmse(L.predict_all(R, train, S, k), R, test) for k in ks]
    checks.append((f"RMSE against k = 1,2,5,10,20,40,80 is 1.229, 1.163, 1.059, 0.921, 0.873, 0.862, 0.872 (got {[round(c, 3) for c in curve]})",
                   [round(c, 3) for c in curve] == [1.229, 1.163, 1.059, 0.921, 0.873, 0.862, 0.872] and int(np.argmin(curve)) == 5))
    hr = (L.random_hit_rate(R, train, test), L.hit_rate(L.popularity_score(train), R, train, test), L.hit_rate(Pu, R, train, test), L.hit_rate(Pi, R, train, test))
    checks.append((f"hit rate@5: random 0.271, popularity 0.747, user-based 0.621, item-based 0.678 (got {hr[0]:.3f}, {hr[1]:.3f}, {hr[2]:.3f}, {hr[3]:.3f})",
                   tuple(round(h, 3) for h in hr) == (0.271, 0.747, 0.621, 0.678)))
    checks.append(("every method beats random; popularity is a strong baseline on this exposure-biased test", min(hr[1:]) > hr[0] and hr[1] > hr[2]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
